# Diagnose weak lip sync

**The symptom:** the mouth barely opens and does not track the words.

There are two suspects, and both come from instructions I gave you:

1. **`num_frames=8`.** I halved it from the upstream 16 so 512 would fit a 15 GB
   card. `num_frames` sets the audio context per window, and LatentSync's paper
   names the frame count as a critical factor. Halving it may have cost more sync
   quality than it bought in memory.

2. **Your base footage is mouth-closed and silent.** I told you to record that
   way to avoid double-articulation. But LatentSync has a documented failure mode
   called *lip shape leakage* — the source mouth shape bleeds into the output — and
   its own demo assets are people **talking**. A rigidly closed source mouth
   plausibly biases a leaky model toward keeping it closed.

Rather than guess a third time, this runs a 2×2 and lets you look:

|                       | `num_frames=8` | `num_frames=16` |
|-----------------------|----------------|-----------------|
| **your footage**      | A              | B               |
| **upstream demo**     | C              | D               |

- **C and D sync well, A and B don't** → your footage is the problem. Re-record
  while speaking.
- **B and D beat A and C** → `num_frames` is the problem. Use 16, which means the
  256 crop and the 720p profile.
- **Both matter** → likely, and then you need 16 *and* talking footage.
- **Nothing syncs well** → LatentSync is the wrong model for this face. Switch to
  MuseTalk, which is a different architecture.

Runs at the **256 crop** so all four fit in memory and finish quickly — this is
about sync, and crop size affects sharpness, not timing.

Needs GPU + Internet + the `talkinghead-assets` dataset attached.

## 1. Setup

Uses the **256** checkpoint (LatentSync 1.5), which is a separate ~5 GB download
from the 1.6 weights the spike used. 1.6 was trained at 512 and 1.5 at 256;
running one at the other's resolution is a mismatch.

In [ ]:
import os, subprocess, sys, time, copy
from pathlib import Path

import torch
assert torch.cuda.is_available(), "Set Accelerator to GPU and re-run."
print(torch.cuda.get_device_name(0))

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "LatentSync"
PINNED = "a229c3948406bc2cf6eaf4873e662e70c6a04746"

if not (REPO_DIR / "scripts" / "inference.py").is_file():
    subprocess.run(["git", "clone", "--quiet",
                    "https://github.com/bytedance/LatentSync", str(REPO_DIR)], check=True)
    subprocess.run(["git", "checkout", "--quiet", PINNED], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print("repo ready at", PINNED[:12])

In [ ]:
# Same list the spike converged on. Installed individually because one
# unresolvable pin aborts the whole pip command and takes everything with it.
PACKAGES = [
    "diffusers==0.32.2", "transformers==4.48.0", "decord==0.6.0", "accelerate",
    "einops", "omegaconf", "opencv-python", "mediapipe", "python_speech_features",
    "librosa", "scenedetect", "ffmpeg-python", "imageio", "imageio-ffmpeg",
    "lpips", "face-alignment", "kornia", "insightface==0.7.3", "onnxruntime-gpu",
    "DeepCache==0.1.1", "soundfile",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cython"],
               capture_output=True)
for pkg in PACKAGES:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  FAILED {pkg}")

# Definitive check: exercises every module-level import of the real entry point.
probe = subprocess.run([sys.executable, "-m", "scripts.inference", "--help"],
                       cwd=REPO_DIR, capture_output=True, text=True)
print("imports OK" if probe.returncode == 0
      else "BROKEN:\n" + "\n".join(probe.stderr.splitlines()[-15:]))

In [ ]:
from huggingface_hub import snapshot_download

CKPT_DIR = REPO_DIR / "checkpoints"
if not (CKPT_DIR / "latentsync_unet.pt").is_file():
    snapshot_download(repo_id="ByteDance/LatentSync-1.5", local_dir=str(CKPT_DIR))

UNET = CKPT_DIR / "latentsync_unet.pt"
print(f"checkpoint: {UNET.stat().st_size / 1024**3:.1f} GB")

# Cache this for later. Downloading 5-10 GB every session is most of the 30
# minutes you are seeing -- see the last cell.
print("\nweights present:", sorted(p.name for p in CKPT_DIR.iterdir()))

## 2. Build the two source clips

Both driven by the **same audio**, so any difference in sync is attributable to
the footage rather than the speech.

In [ ]:
DIAG = WORK / "diagnose"
DIAG.mkdir(exist_ok=True)

yours = next(Path("/kaggle/input").rglob("base_loop.mp4"), None)
if yours is None:
    raise SystemExit("Attach the talkinghead-assets dataset via + Add Input.")
print("your footage:  ", yours)

# LatentSync's own demo clip -- a person actually speaking. This is the control.
demo = next((REPO_DIR / "assets").rglob("demo1_video.mp4"), None) \
    or next((REPO_DIR / "assets").rglob("*.mp4"), None)
if demo is None:
    raise SystemExit(f"No demo video in {REPO_DIR / 'assets'}")
print("upstream demo: ", demo)

CLIP_SECONDS = 8   # long enough to judge sync, short enough to iterate

SRC = {}
for label, src in (("yours", yours), ("demo", demo)):
    dst = DIAG / f"src_{label}.mp4"
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(src),
        "-t", str(CLIP_SECONDS), "-an",
        "-c:v", "libx264", "-crf", "16", "-pix_fmt", "yuv420p", str(dst),
    ], check=True)
    SRC[label] = dst

# Real speech, not a tone -- a tone gives the model nothing to articulate and
# would make every cell look broken.
AUDIO = DIAG / "drive.wav"
demo_audio = next((REPO_DIR / "assets").rglob("demo1_audio.wav"), None) \
    or next((REPO_DIR / "assets").rglob("*.wav"), None) \
    or next(Path("/kaggle/input").rglob("reference.wav"), None)
print("driving audio: ", demo_audio)
subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error", "-i", str(demo_audio),
    "-t", str(CLIP_SECONDS), "-ac", "1", "-ar", "16000", str(AUDIO),
], check=True)

for label, path in SRC.items():
    out = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
                          "-show_entries", "stream=width,height,nb_frames",
                          "-of", "csv=p=0", str(path)],
                         capture_output=True, text=True).stdout.strip()
    print(f"  {label:6s} {out}")

## 3. Run the 2×2

Four renders at the 256 crop. Fixed seed, so differences come from the variables
and not from sampling noise.

In [ ]:
import yaml

base_cfg = yaml.safe_load((REPO_DIR / "configs" / "unet" / "stage2.yaml").read_text())
env = dict(os.environ, PYTORCH_ALLOC_CONF="expandable_segments:True")

results = []
for footage in ("yours", "demo"):
    for num_frames in (8, 16):
        cfg = copy.deepcopy(base_cfg)
        cfg["data"]["resolution"] = 256
        cfg["data"]["num_frames"] = num_frames
        cfg_path = DIAG / f"unet_256_nf{num_frames}.yaml"
        cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

        out = DIAG / f"out_{footage}_nf{num_frames}.mp4"
        cmd = [
            sys.executable, "-m", "scripts.inference",
            "--unet_config_path", str(cfg_path),
            "--inference_ckpt_path", str(UNET),
            "--inference_steps", "20",
            "--guidance_scale", "1.5",
            "--seed", "1247",
            "--enable_deepcache",
            "--video_path", str(SRC[footage]),
            "--audio_path", str(AUDIO),
            "--video_out_path", str(out),
        ]
        print(f"--- {footage}, num_frames={num_frames} ---")
        t0 = time.time()
        p = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, env=env)
        dt = time.time() - t0
        ok = p.returncode == 0 and out.exists()
        results.append({"footage": footage, "num_frames": num_frames,
                        "out": out, "ok": ok, "secs": dt})
        print(f"    {'ok' if ok else 'FAILED'} in {dt:.0f}s")
        if not ok:
            print("\n".join(p.stderr.strip().splitlines()[-12:]))

print()
for r in results:
    print(f"  {r['footage']:6s} nf={r['num_frames']:2d}  "
          f"{'ok' if r['ok'] else 'FAILED':6s}  {r['secs']:5.0f}s")

## 4. Watch them

Each is cropped to the mouth region and enlarged, because at full frame a subtle
articulation failure is invisible.

**What to look for:** does the jaw actually drop on open vowels? Do the lips
close on `m`, `b`, `p`? A mouth that merely blurs or twitches is the failure mode
you are diagnosing.

In [ ]:
from IPython.display import Video, display, HTML

for r in results:
    if not r["ok"]:
        continue
    zoom = r["out"].with_name(r["out"].stem + "_mouth.mp4")
    # Lower-middle third of the frame, scaled up. Approximate but consistent
    # across all four, which is what makes them comparable.
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(r["out"]),
        "-vf", "crop=iw/2:ih/3:iw/4:ih/2,scale=560:-2", "-an", str(zoom),
    ], check=True)
    display(HTML(f"<h4>{r['footage']} — num_frames={r['num_frames']}</h4>"))
    display(Video(str(zoom), embed=True, width=560))

## 5. Reading the result, and the speed problem

### Which variable mattered

| Observation | Conclusion | Action |
|---|---|---|
| demo syncs, yours doesn't | your footage | Re-record **while talking** |
| `nf=16` beats `nf=8` in both | `num_frames` | Use 16 → the 256 crop → `TH_PROFILE=720p` |
| both effects visible | both | 16 **and** talking footage |
| nothing syncs | LatentSync is wrong here | Try MuseTalk |

### If you need to re-record

My original "mouth closed, do not speak" was wrong for this model family.
Corrected spec:

- **Talk naturally through the whole take.** Read anything — the content is
  irrelevant, since the mouth gets repainted. What matters is that your jaw and
  lips are *already moving*, which is what LatentSync's demo footage does.
- Everything else stands: 1080p, tripod, mid shot, even frontal light, plain
  background, eyes on the lens, 60–90 seconds.
- Keep the audio in the recording. The pipeline strips it, but you may want it.

### About the 30 minutes

Very little of that is inference. Roughly:

| Stage | Time | Fixable? |
|---|---|---|
| Weight download (9.8 GB) | 8–12 min | **Yes — cache it** |
| pip installs | 4–6 min | Partly |
| Inference (5 s clip × 3 sweep configs) | 3–4 min | Yes — stop sweeping |
| Model load, face detection, encode | 2–3 min | Not really |

**The one that matters: cache the weights.** Run the cell below, download
`checkpoints.tar`, and upload it as a second private Kaggle Dataset. Every later
session mounts it instead of downloading, which removes 8–12 minutes from *every*
run. `talkinghead.lipsync.latentsync.LatentSync` already prefers a `cache_dir`
over downloading.

Then, in order of remaining impact:

1. **`inference_steps` 20 → 10.** Time scales roughly linearly. Upstream
   documents 20–50, so 10 is below the sanctioned range — check the output, but
   it is the most direct lever on render time.
2. **Use the 256 crop.** A quarter the pixels per frame. Also the configuration
   that may fix your sync, so this is likely free.
3. **Stop re-running the sweep** once you know which config works.
4. **Keep `--enable_deepcache` on.** Already default in the provider.

In [ ]:
# Bundle the weights so they can be uploaded once and mounted forever after.
# Uncompressed on purpose: these are already-compressed tensors, so gzip would
# cost minutes and save almost nothing.
TAR = WORK / "checkpoints.tar"
if not TAR.exists():
    subprocess.run(["tar", "-cf", str(TAR), "-C", str(REPO_DIR), "checkpoints"],
                   check=True)
print(f"{TAR}  ({TAR.stat().st_size / 1024**3:.1f} GB)")
print(
    "\nDownload it from the Output panel, then:\n"
    "  1. kaggle.com/datasets -> New Dataset\n"
    "  2. Upload checkpoints.tar, title it 'latentsync-weights', keep it Private\n"
    "  3. Attach it alongside talkinghead-assets in future sessions\n"
    "  4. Untar once per session into the repo, or point cache_dir at the mount\n"
)